# ClauseXplain v6 (ledgar): 905 Indian clauses (v4 + batch 3 + batch 4), document-held-out
Same protocol and settings as `indian_v5_kaggle.ipynb` (8 epochs, no tuning on these results); one run per notebook
(`v6_legalbert_cuad_indian_ledgar_s13`), so the plain and LEDGAR notebooks can run at the same time. Batch 4 template groups (dhoot,
ssa_jan2025) are held out together. The summary also scores the 334 v4 clauses alone, so the result compares with the
v4 and v5 runs on the same test clauses. **Keep this notebook private** (batch 3 clauses name private individuals).

Settings: attach `clausexplain-indian-eval` and `scan-vs-span`; **GPU T4**; **Internet ON** (LegalBERT weights and the
LEDGAR parquet, ~21 MB). Run all; download `indian_v6_ledgar_results.zip`. About 2 h. Resumable.

In [1]:
import os, shutil, base64, io, zipfile, subprocess, sys, json
BASE = "/kaggle/input/datasets/riyasuryawanshi"
if not os.path.exists(BASE): BASE = "/kaggle/input"
EVAL_SRC, MODEL_SRC = f"{BASE}/clausexplain-indian-eval", f"{BASE}/scan-vs-span"
WORK, OUT = "/kaggle/working/clausexplain", "/kaggle/working/v6_runs"
LB_RUN = "/kaggle/working/legalbert_cuad"
LEGALBERT, LONGFORMER = "nlpaueb/legal-bert-base-uncased", "allenai/longformer-base-4096"
if os.path.exists(WORK): shutil.rmtree(WORK)
shutil.copytree(EVAL_SRC, WORK)
zipfile.ZipFile(io.BytesIO(base64.b64decode("<embedded data zip removed: contains private / unmasked clause text; rebuild with notebooks/make_*.py>"))).extractall(WORK)
os.makedirs(OUT, exist_ok=True)
assert os.path.exists(f"{MODEL_SRC}/clauses_mapped.jsonl"), "attach the scan-vs-span dataset"
from transformers import AutoTokenizer
AutoTokenizer.from_pretrained(LEGALBERT)   # fails here if Internet is off
LEDGAR = "/kaggle/working/ledgar_mapped.jsonl"
r = subprocess.run([sys.executable, "ledgar_extra.py", "--out", LEDGAR, "--parquet", "/kaggle/working/ledgar_train.parquet"],
                   capture_output=True, text=True, cwd=WORK); print(r.stdout, r.stderr[-1500:])
assert os.path.exists(LEDGAR)
LB = ["--encoder", LEGALBERT, "--init_ckpt", f"{LB_RUN}/ckpt_cls_only_fold0.pt"]
LG = ["--extra_jsonl", LEDGAR, "--extra_cap", "100"]
RUNS = [("v6_legalbert_cuad_indian_ledgar_s13", "neural", LB + LG + ["--seed", "13"])]
print("ready;", len(RUNS), "configuration:", RUNS[0][0])

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

24737 of 60000 LEDGAR train provisions mapped -> /kaggle/working/ledgar_mapped.jsonl
  Other                          5770
  Modification                   5689
  Governing Law / Jurisdiction   4265
  Payment                        2523
  Assignment                     1802
  Indemnification                1207
  Termination                    1112
  Confidentiality                1012
  Warranty                       499
  Intellectual Property          473
  Dispute Resolution             385
 
ready; 1 configuration: v6_legalbert_cuad_indian_ledgar_s13


In [2]:
# LegalBERT trained on CUAD fold 0 (same split and settings as the zero-shot LegalBERT run), ~4 min.
os.chdir(WORK)
if not os.path.exists(f"{LB_RUN}/ckpt_cls_only_fold0.pt"):
    os.makedirs(f"{LB_RUN}/partial", exist_ok=True)
    shutil.copy(f"{WORK}/partial/splits.json", f"{LB_RUN}/partial/splits.json")
    LF = json.load(open(f"{WORK}/partial/config.json"))
    cmd = [sys.executable, "train_phase3.py", "--cuad_json", f"{MODEL_SRC}/CUAD_v1.json",
           "--mapped_jsonl", f"{MODEL_SRC}/clauses_mapped.jsonl", "--out_dir", LB_RUN, "--encoder", LEGALBERT,
           "--experiments", "cls_only", "--cv_folds", "5", "--folds", "0", "--max_len_span", "512",
           "--pad_multiple", "64", "--resume", "--save_ckpt", "--skip_baselines"]
    for k in ["epochs", "lr", "head_lr", "seed", "test_frac", "val_frac", "bs_clause", "grad_accum", "class_weights", "max_len_clause"]:
        if LF.get(k) is not None: cmd += [f"--{k}", str(LF[k])]
    r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout[-800:], r.stderr[-1500:] if r.returncode else "")
print("LegalBERT CUAD checkpoint:", os.path.exists(f"{LB_RUN}/ckpt_cls_only_fold0.pt"))

70 risk_labelled=   0 categories=12
fold 3: split sizes / category balance / risk-labelled counts |   train  n= 7058 risk_labelled=   0 categories=12 |   val    n=  877 risk_labelled=   0 categories=12 |   test   n= 1851 risk_labelled=   0 categories=12
fold 4: split sizes / category balance / risk-labelled counts |   train  n= 7093 risk_labelled=   0 categories=12 |   val    n=  854 risk_labelled=   0 categories=12 |   test   n= 1839 risk_labelled=   0 categories=12
=== cls_only | fold 1/5 ===
  [cls_only fold0] epoch 1/2 loss=0.9804 val_composite=0.8891 (96s)
  -> ~3 min for this run (upper bound; early stopping may cut it)
  [cls_only fold0] epoch 2/2 loss=0.2881 val_composite=0.8817 (198s)

Wrote /kaggle/working/legalbert_cuad/results.json and /kaggle/working/legalbert_cuad/results.md
 
LegalBERT CUAD checkpoint: True


In [3]:
os.chdir(WORK)
for name, method, extra in RUNS:
    if os.path.exists(f"{OUT}/{name}/metrics.json"):
        print("skip (done):", name); continue
    cmd = [sys.executable, "finetune_indian.py", "run", "--method", method, "--name", name,
           "--indian_dir", f"{WORK}/built_v6", "--judgments_dir", f"{WORK}/built_judgments", "--out_dir", OUT] + extra
    print("=== ", name, flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if line.startswith(("fold", name)) or "Error" in line or "error" in line: print(line, end="", flush=True)
    print("exit", p.wait())

===  v6_legalbert_cuad_indian_ledgar_s13
fold  0 azure                                                   n= 54 acc=0.667 last-epoch loss 0.015 (340s)
fold  1 dec2022                                                 n= 47 acc=0.702 last-epoch loss 0.017 (673s)
fold  2 dhoot                                                   n= 19 acc=0.474 last-epoch loss 0.017 (1013s)
fold  3 indian_aera_rfp_consultant_2025                         n= 10 acc=0.700 last-epoch loss 0.016 (1353s)
fold  4 indian_concord_sha_amendment_2021                       n=  8 acc=0.875 last-epoch loss 0.016 (1696s)
fold  5 indian_cpwd_gcc_2023                                    n= 39 acc=0.385 last-epoch loss 0.015 (2034s)
fold  6 indian_cyient_master_services_2022                      n= 44 acc=0.773 last-epoch loss 0.018 (2371s)
fold  7 indian_dot_unified_licence_agreement                    n= 21 acc=0.714 last-epoch loss 0.025 (2711s)
fold  8 indian_gem_general_terms_and_conditions                 n= 14 acc=0.571 l

In [4]:
r = subprocess.run([sys.executable, "finetune_indian.py", "summarise", "--out_dir", OUT], capture_output=True, text=True, cwd=WORK)
print(r.stdout, r.stderr[-1000:])
# the 334 v4 clauses alone (same test clauses as the v4 and v5 runs); the rest only add training data there
from sklearn.metrics import f1_score
import csv
V4 = {json.loads(l)["clause_id"] for l in open(f"{WORK}/built_v4/indian_mapped.jsonl", encoding="utf-8") if l.strip()}
lines = ["", "## v4 clauses only (n=334) and the added batches", "", "| run | v4 acc | v4 macro-F1 | added acc | added macro-F1 |", "|---|---|---|---|---|"]
for name, _, _ in RUNS:
    p = f"{OUT}/{name}/predictions.csv"
    if not os.path.exists(p): continue
    rs = [r for r in csv.DictReader(open(p, encoding="utf-8")) if r["group"] != "judgment"]; cells = []
    for sub in ([r for r in rs if r["clause_id"] in V4], [r for r in rs if r["clause_id"] not in V4]):
        y, q = [r["true_category"] for r in sub], [r["pred_category"] for r in sub]
        cells += [f"{sum(a == b for a, b in zip(y, q)) / len(y):.3f} (n={len(y)})", f"{f1_score(y, q, average='macro', labels=sorted(set(y))):.3f}"]
    lines.append(f"| {name} | " + " | ".join(cells) + " |")
open(f"{OUT}/summary.md", "a", encoding="utf-8").write("\n".join(lines) + "\n"); print("\n".join(lines))

# Indian set: document-held-out training (leave-one-document-group-out, 12 folds)

Every clause is predicted by a model that never saw its document. Zero-shot CUAD models (no Indian training)
scored 0.347 acc / 0.338 macro-F1 (Longformer) on the 192 seen-category clauses only.

| Run | all acc (95% CI) | all macro-F1 (95% CI) | seen acc / mF1 | unseen acc / mF1 | original / batch 2 acc | judgments acc |
|---|---|---|---|---|---|---|
| v6_legalbert_cuad_indian_ledgar_s13 | 0.650 (0.618-0.680) | 0.643 (0.602-0.676) | 0.589 / 0.615 | 0.755 / 0.838 | 0.671 / 0.491 | 0.636 |
 

## v4 clauses only (n=334) and the added batches

| run | v4 acc | v4 macro-F1 | added acc | added macro-F1 |
|---|---|---|---|---|
| v6_legalbert_cuad_indian_ledgar_s13 | 0.638 (n=334) | 0.574 | 0.657 (n=571) | 0.625 |


In [5]:
with zipfile.ZipFile("/kaggle/working/indian_v6_ledgar_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{OUT}/summary.md", "summary.md")
    for f in ["results.json", "results.md"]:                  # CUAD fold-0 score of the encoder used as init
        if os.path.exists(f"{LB_RUN}/{f}"): z.write(f"{LB_RUN}/{f}", f"cuad_fold0_{f}")
    for name, _, _ in RUNS:
        for f in ["metrics.json", "predictions.csv"]:
            p = f"{OUT}/{name}/{f}"
            if os.path.exists(p): z.write(p, f"{name}/{f}")
print("download /kaggle/working/indian_v6_ledgar_results.zip from the Output tab")

download /kaggle/working/indian_v6_ledgar_results.zip from the Output tab
